# 🛡️ RiskLens — Transaction Fraud Analysis

This notebook provides a lightweight exploratory analysis of the **RiskLens** transaction fraud dataset. The project is designed for the Razorpay AI Buildathon's **AI Risk Manager** use case.

The goal is to understand the transaction data, identify patterns related to fraud, and prepare the data for a machine-learning based fraud-risk system.

## 🎯 Objectives

- Understand the structure of the transaction dataset
- Check missing values and basic data quality
- Compare fraudulent and genuine transactions
- Explore transaction amount and time patterns
- Prepare observations that support the RiskLens fraud-risk model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

DATA_PATH = '../data/risk_data.csv'
df = pd.read_csv(DATA_PATH)

print('Dataset loaded successfully!')
print('Shape:', df.shape)

In [ ]:
# Preview the data
df.head()

## 1. Dataset Overview

The dataset contains transaction-level information such as transaction amount, payment method, product category, customer age, device used, account age, transaction hour, and the fraud label.

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

df.info()

In [ ]:
# Column names
df.columns.tolist()

## 2. Missing Values

Checking missing values is an important preprocessing step because incomplete transaction information can affect model performance.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

## 3. Fraud Distribution

Fraud datasets are usually imbalanced because genuine transactions are much more common than fraudulent ones. This is important when evaluating a fraud model; accuracy alone should not be the only metric.

In [ ]:
fraud_counts = df['Is Fraudulent'].value_counts().sort_index()
fraud_percent = df['Is Fraudulent'].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    'Count': fraud_counts,
    'Percentage': fraud_percent.round(2)
})

summary.index = ['Genuine', 'Fraudulent']
summary

In [ ]:
plt.figure(figsize=(6, 4))
df['Is Fraudulent'].value_counts().sort_index().plot(kind='bar')
plt.xticks([0, 1], ['Genuine', 'Fraudulent'], rotation=0)
plt.ylabel('Number of Transactions')
plt.title('Genuine vs Fraudulent Transactions')
plt.tight_layout()
plt.show()

## 4. Transaction Amount Analysis

Transaction amount can be an important fraud-risk feature. We compare the typical transaction amounts for genuine and fraudulent transactions.

In [ ]:
amount_summary = df.groupby('Is Fraudulent')['Transaction Amount'].agg(
    ['count', 'mean', 'median', 'min', 'max']
)
amount_summary.index = ['Genuine', 'Fraudulent']
amount_summary

In [ ]:
plt.figure(figsize=(8, 4))
df.boxplot(column='Transaction Amount', by='Is Fraudulent')
plt.suptitle('')
plt.title('Transaction Amount by Fraud Status')
plt.xlabel('Fraud Status (0 = Genuine, 1 = Fraudulent)')
plt.ylabel('Transaction Amount')
plt.tight_layout()
plt.show()

## 5. Transaction Hour Analysis

The time of a transaction can also provide useful information for fraud-risk scoring. Here we compare the number of transactions by hour.

In [ ]:
hour_fraud = pd.crosstab(
    df['Transaction Hour'],
    df['Is Fraudulent'],
    normalize='index'
) * 100

hour_fraud.rename(columns={0: 'Genuine %', 1: 'Fraudulent %'})

In [ ]:
plt.figure(figsize=(10, 4))
fraud_by_hour = df.groupby('Transaction Hour')['Is Fraudulent'].mean() * 100
fraud_by_hour.plot(kind='line', marker='o')
plt.xlabel('Transaction Hour')
plt.ylabel('Fraud Rate (%)')
plt.title('Fraud Rate by Transaction Hour')
plt.xticks(range(24))
plt.tight_layout()
plt.show()

## 6. Categorical Feature Analysis

Payment method and product category may also show different transaction patterns. These features are later transformed by the machine-learning preprocessing pipeline.

In [ ]:
payment_fraud = pd.crosstab(
    df['Payment Method'],
    df['Is Fraudulent'],
    normalize='index'
) * 100

payment_fraud.rename(columns={0: 'Genuine %', 1: 'Fraudulent %'}).round(2)

In [ ]:
category_fraud = pd.crosstab(
    df['Product Category'],
    df['Is Fraudulent'],
    normalize='index'
) * 100

category_fraud.rename(columns={0: 'Genuine %', 1: 'Fraudulent %'}).round(2)

## 7. Key Observations

From this analysis, we can identify patterns that are useful for fraud-risk modeling:

1. Fraud and genuine transactions should be evaluated separately because the target is imbalanced.
2. Transaction amount can provide useful information about transaction risk.
3. Transaction timing can contribute to identifying unusual activity.
4. Categorical variables such as payment method and product category can contain useful patterns.
5. These observations support the use of a preprocessing pipeline followed by a machine-learning classifier.

## 🔗 Connection to RiskLens

The insights from this notebook support the main RiskLens application. The trained model uses transaction features to estimate fraud probability. That probability is converted into a 0–100 risk score, classified into risk levels, and explained using SHAP. The Streamlit dashboard then adds business-impact and cost trade-off analysis to support a final risk decision.

## ✅ Conclusion

This exploratory analysis helps understand the transaction dataset before applying machine learning. The final RiskLens system combines data preprocessing, fraud prediction, risk scoring, SHAP explainability, and business cost analysis in an interactive Streamlit dashboard.